<a href="https://colab.research.google.com/github/iamhaseebakh-blip/House-Price-and-Category-Prediction/blob/main/House_Price_Category_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
df = pd.read_csv("train.csv")

print("Dataset shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values in selected columns:")
selected_columns = [
    "SalePrice", "GrLivArea", "OverallQual", "GarageCars",
    "TotalBsmtSF", "FullBath", "YearBuilt"
]
print(df[selected_columns].isnull().sum())

print("\nBasic statistics:")
df[selected_columns].describe()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice")
plt.title("Living Area vs Sale Price")
plt.xlabel("Above Ground Living Area")
plt.ylabel("Sale Price")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["SalePrice"], kde=True)
plt.title("Distribution of House Prices")
plt.xlabel("Sale Price")
plt.show()

In [ ]:
features = [
    "GrLivArea",
    "OverallQual",
    "GarageCars",
    "TotalBsmtSF",
    "FullBath",
    "YearBuilt"
]

target = "SalePrice"

data = df[features + [target]].copy()

for column in features:
    data[column] = data[column].fillna(data[column].median())

area_limit = data["GrLivArea"].quantile(0.99)
data = data[data["GrLivArea"] <= area_limit].copy()

print("Shape after preprocessing:", data.shape)
data.head()

In [ ]:
low_limit = data["SalePrice"].quantile(0.33)
high_limit = data["SalePrice"].quantile(0.66)

def get_price_category(price):
    if price <= low_limit:
        return "Low"
    elif price <= high_limit:
        return "Medium"
    else:
        return "High"

data["PriceCategory"] = data["SalePrice"].apply(get_price_category)

print("Category limits:")
print("Low/Medium limit:", round(low_limit, 2))
print("Medium/High limit:", round(high_limit, 2))

print("\nCategory counts:")
print(data["PriceCategory"].value_counts())

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=data, x="PriceCategory")
plt.title("Number of Houses in Each Price Category")
plt.xlabel("Price Category")
plt.ylabel("Number of Houses")
plt.show()

In [ ]:
X = data[features]
y_price = data["SalePrice"]
y_category = data["PriceCategory"]

X_train, X_test, y_price_train, y_price_test, y_category_train, y_category_test = train_test_split(
    X,
    y_price,
    y_category,
    test_size=0.20,
    random_state=42,
    stratify=y_category
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")

In [ ]:
  linear_model = LinearRegression()

linear_model.fit(X_train_scaled, y_price_train)

price_predictions = linear_model.predict(X_test_scaled)

print("Linear Regression model trained.")

In [ ]:
mae = mean_absolute_error(y_price_test, price_predictions)
mse = mean_squared_error(y_price_test, price_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_price_test, price_predictions)

print("Linear Regression Results")
print("-------------------------")
print("MAE :", round(mae, 2))
print("MSE :", round(mse, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))

In [ ]:
results = pd.DataFrame({
    "Actual Price": y_price_test.values,
    "Predicted Price": price_predictions
})

print(results.head(10))

plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_price_test, y=price_predictions)
plt.xlabel("Actual Sale Price")
plt.ylabel("Predicted Sale Price")
plt.title("Actual vs Predicted House Prices")

min_price = min(y_price_test.min(), price_predictions.min())
max_price = max(y_price_test.max(), price_predictions.max())
plt.plot([min_price, max_price], [min_price, max_price])
plt.show()

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

random_forest.fit(X_train_scaled, y_category_train)

category_predictions = random_forest.predict(X_test_scaled)

print("Random Forest classifier trained.")

In [ ]:
accuracy = accuracy_score(y_category_test, category_predictions)
precision = precision_score(
    y_category_test,
    category_predictions,
    average="weighted",
    zero_division=0
)
recall = recall_score(
    y_category_test,
    category_predictions,
    average="weighted",
    zero_division=0
)
f1 = f1_score(
    y_category_test,
    category_predictions,
    average="weighted",
    zero_division=0
)

print("Random Forest Classification Results")
print("-------------------------------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-score :", round(f1, 4))

print("\nClassification Report:")
print(classification_report(
    y_category_test,
    category_predictions,
    zero_division=0
))

In [ ]:
cm = confusion_matrix(
    y_category_test,
    category_predictions,
    labels=["Low", "Medium", "High"]
)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Low", "Medium", "High"],
    yticklabels=["Low", "Medium", "High"]
)
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
comparison = pd.DataFrame({
    "Task": [
        "Price Prediction",
        "Price Category Classification"
    ],
    "Model": [
        "Linear Regression",
        "Random Forest"
    ],
    "Main Metric": [
        "R²",
        "Accuracy"
    ],
    "Score": [
        r2,
        accuracy
    ]
})

comparison

In [ ]:
example_predictions = X_test.copy()
example_predictions["Actual Price"] = y_price_test.values
example_predictions["Predicted Price"] = price_predictions
example_predictions["Actual Category"] = y_category_test.values
example_predictions["Predicted Category"] = category_predictions

example_predictions.head(10)